## Đọc và merge 2 file transferable và essential skills lại với nhau

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Đọc dữ liệu
essential = pd.read_csv("../data/raw/onet/essential_skills.csv")
transferable = pd.read_csv("../data/raw/onet/transferable_skills.csv")

# Chỉ sử dụng thang LV
essential_lv = essential[essential["Scale ID"] == "LV"].copy()
transferable_lv = transferable[transferable["Scale ID"] == "LV"].copy()

# Kiểm tra trùng mã kỹ năng giữa hai file
overlap = set(essential_lv["Element ID"]) & set(transferable_lv["Element ID"])
assert len(overlap) == 0, f"Skill IDs bị trùng: {overlap}"

# Kiểm tra trùng cặp nghề - kỹ năng trong từng file
key = ["O*NET-SOC Code", "Element ID"]

assert not essential_lv.duplicated(key).any()
assert not transferable_lv.duplicated(key).any()

# Kiểm tra hai file có cùng tập mã nghề
assert set(essential_lv["O*NET-SOC Code"]) == set(
    transferable_lv["O*NET-SOC Code"]
)

# Kết hợp dữ liệu
skills_lv = pd.concat(
    [essential_lv, transferable_lv],
    ignore_index=True
)

print("Số dòng:", len(skills_lv))
print("Số nghề:", skills_lv["O*NET-SOC Code"].nunique())
print("Số kỹ năng:", skills_lv["Element ID"].nunique())
print("Giá trị Data Value bị thiếu:", skills_lv["Data Value"].isna().sum())

Số dòng: 31850
Số nghề: 910
Số kỹ năng: 35
Giá trị Data Value bị thiếu: 0


## Tạo ma trận

In [2]:
# Mỗi nghề là một dòng, mỗi kỹ năng là một cột
skill_matrix = skills_lv.pivot(index="O*NET-SOC Code", columns="Element ID", values="Data Value")
skill_matrix = skill_matrix.apply(pd.to_numeric, errors="coerce")
print("Kích thước ma trận:", skill_matrix.shape)
print("Số ô thiếu:", skill_matrix.isna().sum().sum())

# Kiểm tra chất lượng ma trận ngay sau khi tạo
assert not skill_matrix.isna().any().any(), "Skill matrix có giá trị thiếu"
assert not skill_matrix.index.duplicated().any(), "Mã nghề bị trùng"
assert not skill_matrix.columns.duplicated().any(), "Mã kỹ năng bị trùng"
assert skill_matrix.shape[0] > 0 and skill_matrix.shape[1] > 0
print("DATA VALIDATION: PASS")

Kích thước ma trận: (910, 35)
Số ô thiếu: 0
DATA VALIDATION: PASS


## Tính độ tương đồng giữa các nghề (Tính cosine similarity)

In [3]:
similarity_array = cosine_similarity(skill_matrix)

similarity_df = pd.DataFrame(
    similarity_array,
    index=skill_matrix.index,
    columns=skill_matrix.index
)

print("Kích thước similarity matrix:", similarity_df.shape)

# Tạo z-score cosine sớm để các hàm phía dưới dùng được khi Run All từ đầu.
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(skill_matrix.astype(float))
scaled_similarity_df = pd.DataFrame(
    cosine_similarity(X_scaled),
    index=skill_matrix.index,
    columns=skill_matrix.index
)

# Tạo centered cosine để so sánh các phương pháp.
X_centered = skill_matrix.astype(float) - skill_matrix.astype(float).mean(axis=0)
centered_similarity_df = pd.DataFrame(
    cosine_similarity(X_centered),
    index=skill_matrix.index,
    columns=skill_matrix.index
)

print("Đã tạo raw, z-score và centered cosine similarity.")


Kích thước similarity matrix: (910, 910)
Đã tạo raw, z-score và centered cosine similarity.


## Tạo danh sách top 10 các nghề mà mình có thể chuyển sang

In [4]:
# Đọc thông tin nghề nghiệp
occupation_data = pd.read_csv("../data/raw/onet/occupation_data.csv")
occupation_data = occupation_data[
    ["O*NET-SOC Code", "Title", "Description"]
].drop_duplicates(subset=["O*NET-SOC Code"])


In [5]:
# Baseline: xem Top 10 theo raw cosine để đối chiếu.
# Hàm career_transition() chính thức được định nghĩa sau calculate_skill_gap.
source_code = "11-1011.00"
raw_top10 = (
    similarity_df[source_code]
    .drop(index=source_code)
    .nlargest(10)
    .rename("raw_cosine")
    .rename_axis("occupation_code")
    .reset_index()
    .merge(
        occupation_data[["O*NET-SOC Code", "Title", "Description"]],
        left_on="occupation_code",
        right_on="O*NET-SOC Code",
        how="left",
        validate="many_to_one"
    )
    .drop(columns=["O*NET-SOC Code"])
)
display(raw_top10)


,occupation_code,raw_cosine,Title,Description
0,11-2022.00,0.992996,Sales Managers,"Plan, direct, or coordinate the actual distrib..."
1,11-2011.00,0.992105,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."
2,11-2021.00,0.991565,Marketing Managers,"Plan, direct, or coordinate marketing policies..."
3,11-3071.04,0.990553,Supply Chain Managers,"Direct or coordinate production, purchasing, w..."
4,15-1299.09,0.988883,Information Technology Project Managers,"Plan, initiate, and manage information technol..."
5,27-2012.03,0.988729,Media Programming Directors,Direct and coordinate activities of personnel ...
6,11-1021.00,0.988706,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
7,11-1011.03,0.988080,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
8,11-3061.00,0.987671,Purchasing Managers,"Plan, direct, or coordinate the activities of ..."
9,13-1082.00,0.987428,Project Management Specialists,"Analyze and coordinate the schedule, timeline,..."


## So sánh skill gap giữa hai nghề

In [6]:
#Tạo bảng tên kỹ năng:
skill_names = (
    skills_lv[["Element ID", "Element Name"]]
    .drop_duplicates(subset=["Element ID"])
    .set_index("Element ID")["Element Name"]
)

In [7]:
# Hàm tính chênh lệch
def calculate_skill_gap(source_code, target_code):
    if source_code not in skill_matrix.index:
        raise ValueError(f"Không tìm thấy nghề nguồn: {source_code}")

    if target_code not in skill_matrix.index:
        raise ValueError(f"Không tìm thấy nghề đích: {target_code}")

    source = skill_matrix.loc[source_code]
    target = skill_matrix.loc[target_code]

    result = pd.DataFrame({
        "skill_name": skill_names.reindex(skill_matrix.columns).values,
        "source_level": source.values,
        "target_level": target.values,
        "level_gap": (target - source).values
    })

    result["gap_type"] = np.select(
        [
            result["level_gap"] > 0,
            result["level_gap"] < 0
        ],
        [
            "Target requires higher level",
            "Target requires lower level"
        ],
        default="Same level"
    )

    return result.sort_values("level_gap", ascending=False).reset_index(drop=True)

In [8]:
#Test với 2 nghề
skill_gap = calculate_skill_gap(
    "11-1011.00",
    "47-5013.00"
)

display(skill_gap)
# level_gap > 0: nghề mục tiêu có mức LV cao hơn ở kỹ năng đó.

# level_gap < 0: nghề mục tiêu có mức LV thấp hơn.

# level_gap = 0: hai nghề có cùng mức LV.
# Lưu ý rằng chênh lệch LV chỉ là để so sánh, không tự động có nghĩa người lao động phải học thêm toàn bộ kỹ năng có gap dương.

,skill_name,source_level,target_level,level_gap,gap_type
0,Equipment Maintenance,0.00,3.25,3.25,Target requires higher level
1,Repairing,0.00,3.12,3.12,Target requires higher level
2,Troubleshooting,0.50,3.12,2.62,Target requires higher level
3,Equipment Selection,0.25,2.50,2.25,Target requires higher level
4,Operations Monitoring,1.50,3.62,2.12,Target requires higher level
5,Installation,0.00,2.12,2.12,Target requires higher level
6,Operation and Control,1.38,3.50,2.12,Target requires higher level
7,Quality Control Analysis,1.12,3.12,2.00,Target requires higher level
8,Programming,1.00,0.88,-0.12,Target requires lower level
9,Science,0.75,0.38,-0.37,Target requires lower level


## Hàm career_transition chính thức
Định nghĩa duy nhất, dùng z-score cosine + skill gap percentile.


In [9]:
def career_transition(source_code, top_n=10):
    """
    Xếp hạng nghề đích dựa trên:
    - raw_cosine: cosine trên LV gốc, dùng để tham khảo
    - zscore_cosine: cosine sau khi chuẩn hóa từng kỹ năng
    - mean_absolute_skill_gap: khoảng cách LV trung bình tuyệt đối

    transition_score là điểm xếp hạng heuristic, KHÔNG phải xác suất.
    """
    if source_code not in skill_matrix.index:
        raise ValueError(f"Occupation code not found: {source_code}")

    if isinstance(top_n, bool) or not isinstance(top_n, int) or top_n < 1:
        raise ValueError("top_n phải là số nguyên >= 1")

    raw_scores = similarity_df[source_code].drop(index=source_code)
    z_scores = scaled_similarity_df[source_code].drop(index=source_code)

    mean_gaps = (
        skill_matrix.sub(skill_matrix.loc[source_code], axis="columns")
        .abs()
        .mean(axis=1)
        .drop(index=source_code)
    )

    results = pd.DataFrame({
        "occupation_code": raw_scores.index,
        "raw_cosine": raw_scores.values,
        "zscore_cosine": z_scores.reindex(raw_scores.index).values,
        "mean_absolute_skill_gap": mean_gaps.reindex(raw_scores.index).values
    })

    occupation_info = (
        occupation_data[["O*NET-SOC Code", "Title", "Description"]]
        .drop_duplicates(subset=["O*NET-SOC Code"])
    )
    results = results.merge(
        occupation_info,
        left_on="occupation_code",
        right_on="O*NET-SOC Code",
        how="left",
        validate="many_to_one"
    ).drop(columns=["O*NET-SOC Code"])

    # Percentile rank: giá trị cao hơn luôn tốt hơn.
    # Gap nhỏ hơn là tốt hơn nên đảo chiều percentile.
    results["similarity_percentile"] = results["zscore_cosine"].rank(
        method="average", pct=True
    )
    results["gap_percentile"] = results["mean_absolute_skill_gap"].rank(
        method="average", pct=True
    )
    results["transition_score"] = (
        0.5 * results["similarity_percentile"]
        + 0.5 * (1 - results["gap_percentile"])
    )

    top_occupations = (
        results.sort_values(
            ["transition_score", "zscore_cosine", "mean_absolute_skill_gap"],
            ascending=[False, False, True]
        )
        .head(top_n)
        .reset_index(drop=True)
    )
    top_occupations.index = top_occupations.index + 1

    skill_gaps = {
        target_code: calculate_skill_gap(source_code, target_code)
        for target_code in top_occupations["occupation_code"]
    }

    return top_occupations, skill_gaps


## Tạo bảng xếp hạng kết hợp

In [10]:
# XẾP HẠNG CHÍNH THỨC: dùng career_transition() duy nhất.
source_code = "11-1011.00"
ranked_transitions, _ = career_transition(source_code=source_code, top_n=10)

display(
    ranked_transitions[
        [
            "occupation_code", "Title", "raw_cosine", "zscore_cosine",
            "mean_absolute_skill_gap", "similarity_percentile",
            "gap_percentile", "transition_score"
        ]
    ]
)


,occupation_code,Title,raw_cosine,zscore_cosine,mean_absolute_skill_gap,similarity_percentile,gap_percentile,transition_score
1,11-2022.00,Sales Managers,0.992996,0.948935,0.618857,0.997800,0.002200,0.997800
2,11-1011.03,Chief Sustainability Officers,0.988080,0.949052,0.672571,0.998900,0.008801,0.995050
3,11-2021.00,Marketing Managers,0.991565,0.939395,0.651143,0.995600,0.005501,0.995050
4,11-9032.00,"Education Administrators, Kindergarten through...",0.984241,0.927818,0.544000,0.990099,0.001100,0.994499
5,11-9151.00,Social and Community Service Managers,0.983405,0.922167,0.671143,0.988999,0.007701,0.990649
6,11-2011.00,Advertising and Promotions Managers,0.992105,0.929015,0.698286,0.991199,0.011001,0.990099
7,11-3071.04,Supply Chain Managers,0.990553,0.948037,0.740000,0.996700,0.017602,0.989549
8,11-9111.00,Medical and Health Services Managers,0.984579,0.920696,0.683143,0.987899,0.009901,0.988999
9,15-1299.09,Information Technology Project Managers,0.988883,0.935908,0.737143,0.993399,0.016502,0.988449
10,11-3121.00,Human Resources Managers,0.987107,0.912737,0.651429,0.983498,0.006601,0.988449


## Xếp hạng kết hợp

Các cell sau dùng duy nhất hàm `career_transition()` đã định nghĩa ở trên. Các bảng raw cosine riêng lẻ chỉ là baseline để so sánh.


## Chạy thử

In [63]:
# raw_cosine: độ tương đồng cosine trên mức kỹ năng LV gốc.

# zscore_cosine: độ tương đồng sau khi chuẩn hóa mức kỹ năng theo từng kỹ năng.

# mean_absolute_skill_gap: độ chênh lệch LV tuyệt đối trung bình giữa hai nghề; càng thấp càng gần.

# transition_score: điểm xếp hạng kết hợp độ tương đồng chuẩn hóa và khoảng cách kỹ năng; không phải xác suất thành công khi chuyển nghề.

top_10, skill_gaps = career_transition(
    source_code="15-1252.00",
    top_n=10
)

display(
    top_10[
        [
            "occupation_code",
            "Title",
            "raw_cosine",
            "zscore_cosine",
            "mean_absolute_skill_gap",
            "transition_score"
        ]
    ]
)

,occupation_code,Title,raw_cosine,zscore_cosine,mean_absolute_skill_gap,transition_score
1,15-1243.00,Database Architects,0.994121,0.936816,0.248000,0.999450
2,15-1299.07,Blockchain Engineers,0.988664,0.893643,0.310286,0.998350
3,15-1254.00,Web Developers,0.990058,0.875155,0.338000,0.996700
4,15-1242.00,Database Administrators,0.989139,0.858294,0.346571,0.995050
5,15-1243.01,Data Warehousing Specialists,0.989816,0.855322,0.372571,0.993949
6,15-1211.00,Computer Systems Analysts,0.988962,0.887494,0.418286,0.992299
7,15-1299.08,Computer Systems Engineers/Architects,0.985526,0.862635,0.417429,0.991749
8,15-1299.01,Web Administrators,0.986203,0.837357,0.386000,0.991749
9,15-1299.05,Information Security Engineers,0.985580,0.834934,0.388571,0.990649
10,15-1253.00,Software Quality Assurance Analysts and Testers,0.986198,0.841132,0.435429,0.988449


## Xem skill gap

In [64]:
target_code = top_10.iloc[0]["occupation_code"]

display(skill_gaps[target_code])

,skill_name,source_level,target_level,level_gap,gap_type
0,Complex Problem Solving,3.50,4.00,0.50,Target requires higher level
1,Instructing,3.12,3.62,0.50,Target requires higher level
2,Writing,3.62,3.88,0.26,Target requires higher level
3,Speaking,3.62,3.88,0.26,Target requires higher level
4,Operations Analysis,3.12,3.38,0.26,Target requires higher level
5,Management of Material Resources,1.00,1.25,0.25,Target requires higher level
6,Management of Financial Resources,1.00,1.25,0.25,Target requires higher level
7,Judgment and Decision Making,3.75,3.88,0.13,Target requires higher level
8,Management of Personnel Resources,2.62,2.75,0.13,Target requires higher level
9,Monitoring,3.50,3.62,0.12,Target requires higher level


## Test với nhiều nghề nguồn

In [65]:
test_occupations = {
    "11-1011.00": "Chief Executives",
    "15-1252.00": "Software Developers",
    "13-2011.00": "Accountants and Auditors",
}

for code, name in test_occupations.items():
    print(f"\n{'=' * 70}")
    print(f"Nghề nguồn: {name} ({code})")

    try:
        top_10, skill_gaps = career_transition(
            source_code=code,
            top_n=10
        )

        display(
            top_10[
                [
                    "occupation_code",
                    "Title",
                    "raw_cosine",
                    "zscore_cosine",
                    "mean_absolute_skill_gap",
                    "transition_score"
                ]
            ]
        )

        # Kiểm tra các điều kiện cơ bản
        assert len(top_10) == 10
        assert code not in top_10["occupation_code"].values
        assert top_10["occupation_code"].is_unique

        print("Kiểm tra: PASS")

    except Exception as e:
        print(f"Kiểm tra: FAIL — {e}")


Nghề nguồn: Chief Executives (11-1011.00)


,occupation_code,Title,raw_cosine,zscore_cosine,mean_absolute_skill_gap,transition_score
1,11-2022.00,Sales Managers,0.992996,0.948935,0.618857,0.997800
2,11-1011.03,Chief Sustainability Officers,0.988080,0.949052,0.672571,0.995050
3,11-2021.00,Marketing Managers,0.991565,0.939395,0.651143,0.995050
4,11-9032.00,"Education Administrators, Kindergarten through...",0.984241,0.927818,0.544000,0.994499
5,11-9151.00,Social and Community Service Managers,0.983405,0.922167,0.671143,0.990649
6,11-2011.00,Advertising and Promotions Managers,0.992105,0.929015,0.698286,0.990099
7,11-3071.04,Supply Chain Managers,0.990553,0.948037,0.740000,0.989549
8,11-9111.00,Medical and Health Services Managers,0.984579,0.920696,0.683143,0.988999
9,15-1299.09,Information Technology Project Managers,0.988883,0.935908,0.737143,0.988449
10,11-3121.00,Human Resources Managers,0.987107,0.912737,0.651429,0.988449


Kiểm tra: PASS

Nghề nguồn: Software Developers (15-1252.00)


,occupation_code,Title,raw_cosine,zscore_cosine,mean_absolute_skill_gap,transition_score
1,15-1243.00,Database Architects,0.994121,0.936816,0.248000,0.999450
2,15-1299.07,Blockchain Engineers,0.988664,0.893643,0.310286,0.998350
3,15-1254.00,Web Developers,0.990058,0.875155,0.338000,0.996700
4,15-1242.00,Database Administrators,0.989139,0.858294,0.346571,0.995050
5,15-1243.01,Data Warehousing Specialists,0.989816,0.855322,0.372571,0.993949
6,15-1211.00,Computer Systems Analysts,0.988962,0.887494,0.418286,0.992299
7,15-1299.08,Computer Systems Engineers/Architects,0.985526,0.862635,0.417429,0.991749
8,15-1299.01,Web Administrators,0.986203,0.837357,0.386000,0.991749
9,15-1299.05,Information Security Engineers,0.985580,0.834934,0.388571,0.990649
10,15-1253.00,Software Quality Assurance Analysts and Testers,0.986198,0.841132,0.435429,0.988449


Kiểm tra: PASS

Nghề nguồn: Accountants and Auditors (13-2011.00)


,occupation_code,Title,raw_cosine,zscore_cosine,mean_absolute_skill_gap,transition_score
1,41-3031.00,"Securities, Commodities, and Financial Service...",0.993297,0.866427,0.242000,0.999450
2,13-1041.08,Customs Brokers,0.991213,0.848122,0.275143,0.998350
3,11-2033.00,Fundraising Managers,0.991229,0.832433,0.318571,0.995050
4,13-1199.06,Online Merchants,0.990382,0.793709,0.303714,0.994499
5,13-2031.00,Budget Analysts,0.985763,0.805476,0.314286,0.993949
6,13-2052.00,Personal Financial Advisors,0.988487,0.813618,0.320857,0.993399
7,29-2099.08,Patient Representatives,0.985879,0.752022,0.300857,0.992299
8,17-3011.00,Architectural and Civil Drafters,0.985038,0.754742,0.307143,0.991749
9,13-1051.00,Cost Estimators,0.987482,0.827525,0.337429,0.990649
10,13-1131.00,Fundraisers,0.990242,0.776957,0.338857,0.987349


Kiểm tra: PASS


## Giải thích vì sao hệ thống đề xuất 1 nghề

In [66]:
def explain_transition(source_code, target_code, top_n=5):
    if source_code not in skill_matrix.index:
        raise ValueError(f"Occupation code not found: {source_code}")
    if target_code not in skill_matrix.index:
        raise ValueError(f"Target occupation code not found: {target_code}")

    gap = calculate_skill_gap(source_code, target_code).copy()
    raw_similarity = float(similarity_df.loc[source_code, target_code])
    zscore_similarity = float(scaled_similarity_df.loc[source_code, target_code])
    mean_absolute_skill_gap = float(gap["level_gap"].abs().mean())

    skills_to_focus = (
        gap[gap["level_gap"] > 0]
        .sort_values("level_gap", ascending=False)
        .head(top_n)
    )

    transferable = (
        gap.assign(abs_gap=gap["level_gap"].abs())
        .sort_values(["abs_gap", "skill_name"], ascending=[True, True])
        .head(top_n)
        .drop(columns=["abs_gap"])
    )

    return {
        "similarity": raw_similarity,  # giữ tương thích với code cũ
        "raw_cosine": raw_similarity,
        "zscore_cosine": zscore_similarity,
        "mean_absolute_skill_gap": mean_absolute_skill_gap,
        "skills_to_focus": skills_to_focus,
        "transferable_skills": transferable
    }


In [67]:
source_code = "15-1252.00"

top_10, skill_gaps = career_transition(
    source_code=source_code,
    top_n=10
)

target_code = top_10.iloc[0]["occupation_code"]

explanation = explain_transition(source_code, target_code)

print("Nghề nguồn:", source_code)
print("Nghề mục tiêu:", target_code)
print(
    "Cosine similarity:",
    round(explanation["similarity"], 4)
)
print(
    "Mean absolute skill gap:",
    round(explanation["mean_absolute_skill_gap"], 4)
)

print("\nCác kỹ năng cần chú ý:")
display(explanation["skills_to_focus"])

print("\nCác kỹ năng có mức LV tương tự:")
display(explanation["transferable_skills"])

Nghề nguồn: 15-1252.00
Nghề mục tiêu: 15-1243.00
Cosine similarity: 0.9941
Mean absolute skill gap: 0.248

Các kỹ năng cần chú ý:


,skill_name,source_level,target_level,level_gap,gap_type
0,Complex Problem Solving,3.50,4.00,0.50,Target requires higher level
1,Instructing,3.12,3.62,0.50,Target requires higher level
2,Writing,3.62,3.88,0.26,Target requires higher level
3,Speaking,3.62,3.88,0.26,Target requires higher level
4,Operations Analysis,3.12,3.38,0.26,Target requires higher level



Các kỹ năng có mức LV tương tự:


,skill_name,source_level,target_level,level_gap,gap_type
13,Active Learning,3.62,3.62,0.0,Same level
10,Coordination,3.12,3.12,0.0,Same level
11,Negotiation,2.88,2.88,0.0,Same level
14,Programming,4.12,4.12,0.0,Same level
12,Service Orientation,3.00,3.00,0.0,Same level


## Kiểm tra chất lượng dữ liệu và similarity

In [68]:
import numpy as np

print("=== SIMILARITY VALIDATION ===")
print("Kích thước:", similarity_df.shape)
print("Giá trị thiếu:", similarity_df.isna().sum().sum())

# Ma trận phải đối xứng
print(
    "Sai lệch đối xứng lớn nhất:",
    np.abs(similarity_df.values - similarity_df.values.T).max()
)

# Mỗi nghề phải có similarity bằng 1 với chính nó
print(
    "Sai lệch đường chéo lớn nhất:",
    np.abs(np.diag(similarity_df.values) - 1).max()
)

assert similarity_df.shape == (910, 910)
assert np.isfinite(similarity_df.values).all()
assert np.allclose(similarity_df.values, similarity_df.values.T)
assert np.allclose(np.diag(similarity_df.values), 1)

print("SIMILARITY VALIDATION: PASS")

=== SIMILARITY VALIDATION ===
Kích thước: (910, 910)
Giá trị thiếu: 0
Sai lệch đối xứng lớn nhất: 0.0
Sai lệch đường chéo lớn nhất: 8.881784197001252e-16
SIMILARITY VALIDATION: PASS


In [69]:
test_codes = [
    "11-1011.00",
    "15-1252.00",
    "13-2011.00"
]

for code in test_codes:
    top_10, _ = career_transition(code, top_n=10)

    print(f"\nNghề nguồn: {code}")
    print("Raw cosine trung bình top 10:",
          round(top_10["raw_cosine"].mean(), 4))
    print("Gap tuyệt đối trung bình top 10:",
          round(top_10["mean_absolute_skill_gap"].mean(), 4))

    display(
        top_10[
            ["occupation_code", "Title",
             "raw_cosine", "zscore_cosine",
             "mean_absolute_skill_gap",
             "transition_score"]
        ]
    )


Nghề nguồn: 11-1011.00
Raw cosine trung bình top 10: 0.9884
Gap tuyệt đối trung bình top 10: 0.6668


,occupation_code,Title,raw_cosine,zscore_cosine,mean_absolute_skill_gap,transition_score
1,11-2022.00,Sales Managers,0.992996,0.948935,0.618857,0.997800
2,11-1011.03,Chief Sustainability Officers,0.988080,0.949052,0.672571,0.995050
3,11-2021.00,Marketing Managers,0.991565,0.939395,0.651143,0.995050
4,11-9032.00,"Education Administrators, Kindergarten through...",0.984241,0.927818,0.544000,0.994499
5,11-9151.00,Social and Community Service Managers,0.983405,0.922167,0.671143,0.990649
6,11-2011.00,Advertising and Promotions Managers,0.992105,0.929015,0.698286,0.990099
7,11-3071.04,Supply Chain Managers,0.990553,0.948037,0.740000,0.989549
8,11-9111.00,Medical and Health Services Managers,0.984579,0.920696,0.683143,0.988999
9,15-1299.09,Information Technology Project Managers,0.988883,0.935908,0.737143,0.988449
10,11-3121.00,Human Resources Managers,0.987107,0.912737,0.651429,0.988449



Nghề nguồn: 15-1252.00
Raw cosine trung bình top 10: 0.9884
Gap tuyệt đối trung bình top 10: 0.3661


,occupation_code,Title,raw_cosine,zscore_cosine,mean_absolute_skill_gap,transition_score
1,15-1243.00,Database Architects,0.994121,0.936816,0.248000,0.999450
2,15-1299.07,Blockchain Engineers,0.988664,0.893643,0.310286,0.998350
3,15-1254.00,Web Developers,0.990058,0.875155,0.338000,0.996700
4,15-1242.00,Database Administrators,0.989139,0.858294,0.346571,0.995050
5,15-1243.01,Data Warehousing Specialists,0.989816,0.855322,0.372571,0.993949
6,15-1211.00,Computer Systems Analysts,0.988962,0.887494,0.418286,0.992299
7,15-1299.08,Computer Systems Engineers/Architects,0.985526,0.862635,0.417429,0.991749
8,15-1299.01,Web Administrators,0.986203,0.837357,0.386000,0.991749
9,15-1299.05,Information Security Engineers,0.985580,0.834934,0.388571,0.990649
10,15-1253.00,Software Quality Assurance Analysts and Testers,0.986198,0.841132,0.435429,0.988449



Nghề nguồn: 13-2011.00
Raw cosine trung bình top 10: 0.9889
Gap tuyệt đối trung bình top 10: 0.3059


,occupation_code,Title,raw_cosine,zscore_cosine,mean_absolute_skill_gap,transition_score
1,41-3031.00,"Securities, Commodities, and Financial Service...",0.993297,0.866427,0.242000,0.999450
2,13-1041.08,Customs Brokers,0.991213,0.848122,0.275143,0.998350
3,11-2033.00,Fundraising Managers,0.991229,0.832433,0.318571,0.995050
4,13-1199.06,Online Merchants,0.990382,0.793709,0.303714,0.994499
5,13-2031.00,Budget Analysts,0.985763,0.805476,0.314286,0.993949
6,13-2052.00,Personal Financial Advisors,0.988487,0.813618,0.320857,0.993399
7,29-2099.08,Patient Representatives,0.985879,0.752022,0.300857,0.992299
8,17-3011.00,Architectural and Civil Drafters,0.985038,0.754742,0.307143,0.991749
9,13-1051.00,Cost Estimators,0.987482,0.827525,0.337429,0.990649
10,13-1131.00,Fundraisers,0.990242,0.776957,0.338857,0.987349


In [70]:
source_code = "15-1252.00"
target_code = "15-1243.00"

gap = calculate_skill_gap(source_code, target_code)

expected_gap = (
    skill_matrix.loc[target_code]
    - skill_matrix.loc[source_code]
)

actual_gap = gap.set_index("skill_name")["level_gap"]

print("Số kỹ năng:", len(gap))
print("Sai lệch gap lớn nhất:",
      np.abs(
          actual_gap.reindex(skill_names.values).values
          - expected_gap.reindex(skill_names.index).values
      ).max()
)

Số kỹ năng: 35
Sai lệch gap lớn nhất: 0.0


In [71]:
# Phân phối similarity: bỏ đường chéo (mỗi nghề so với chính nó)
raw_scores = similarity_df.values[np.triu_indices_from(similarity_df.values, k=1)]
centered_scores = centered_similarity_df.values[np.triu_indices_from(centered_similarity_df.values, k=1)]
print("=== RAW COSINE SIMILARITY ===")
print(pd.Series(raw_scores).describe().round(4))
print("\n=== CENTERED COSINE SIMILARITY ===")
print(pd.Series(centered_scores).describe().round(4))

pairs = [("11-1011.00", "11-2022.00"), ("15-1252.00", "15-1243.00"), ("13-2011.00", "41-3031.00")]
for source, target in pairs:
    print(f"\n{source} -> {target}")
    print("Raw cosine:", round(similarity_df.loc[source, target], 4))
    print("Centered cosine:", round(centered_similarity_df.loc[source, target], 4))

=== RAW COSINE SIMILARITY ===
count    413595.0000
mean          0.9229
std           0.0501
min           0.6759
25%           0.8929
50%           0.9369
75%           0.9623
max           0.9980
dtype: float64

=== CENTERED COSINE SIMILARITY ===
count    413595.0000
mean          0.0003
std           0.4991
min          -0.9483
25%          -0.4480
50%          -0.0167
75%           0.4317
max           0.9817
dtype: float64

11-1011.00 -> 11-2022.00
Raw cosine: 0.993
Centered cosine: 0.9353

15-1252.00 -> 15-1243.00
Raw cosine: 0.9941
Centered cosine: 0.933

13-2011.00 -> 41-3031.00
Raw cosine: 0.9933
Centered cosine: 0.9028


In [72]:
# Kiểm tra độ biến thiên của LV trong từng kỹ năng
skill_variation = pd.DataFrame({
    "skill_name": skill_names.reindex(skill_matrix.columns).values,
    "mean_lv": skill_matrix.mean(axis=0).values,
    "std_lv": skill_matrix.std(axis=0).values,
    "min_lv": skill_matrix.min(axis=0).values,
    "max_lv": skill_matrix.max(axis=0).values,
})

display(
    skill_variation.sort_values("std_lv")
    .round(3)
    .reset_index(drop=True)
)

,skill_name,mean_lv,std_lv,min_lv,max_lv
0,Time Management,3.036,0.493,1.62,4.75
1,Coordination,3.141,0.511,1.75,5.00
2,Monitoring,3.381,0.568,1.75,5.25
3,Service Orientation,2.861,0.632,0.62,5.25
4,Active Listening,3.560,0.645,2.00,5.88
5,Critical Thinking,3.581,0.654,1.62,5.75
6,Negotiation,2.546,0.677,0.50,4.88
7,Social Perceptiveness,3.117,0.678,1.62,5.75
8,Judgment and Decision Making,3.200,0.680,1.62,5.38
9,Management of Personnel Resources,2.564,0.691,0.00,5.25


## Chuẩn hóa lại từng skills

In [73]:
# Thống kê z-score cosine đã được tính ở Cell 5; không tính lại ma trận.
scaled_scores = scaled_similarity_df.values[np.triu_indices_from(scaled_similarity_df.values, k=1)]
print("=== Z-SCORE COSINE ===")
print(pd.Series(scaled_scores).describe().round(4))

=== Z-SCORE COSINE ===
count    413595.0000
mean         -0.0003
std           0.5020
min          -0.9484
25%          -0.4510
50%          -0.0152
75%           0.4395
max           0.9879
dtype: float64


## So sánh 3 cách đo similarity

Raw cosine phản ánh hướng của vector LV gốc; centered cosine trừ trung bình mỗi kỹ năng; z-score cosine vừa trừ trung bình vừa chia độ lệch chuẩn. Không chọn phương pháp chỉ vì điểm thấp hơn — hãy so sánh thứ hạng nghề và tính hợp lý của đề xuất.


In [74]:
# So sánh Top 10 theo ba ma trận similarity đã tạo ở Cell 5
def top_occupations_for_metric(similarity_matrix, source_code, top_n=10):
    scores = similarity_matrix[source_code].drop(index=source_code).nlargest(top_n)
    result = scores.rename("similarity").rename_axis("occupation_code").reset_index()
    return result.merge(
        occupation_data[["O*NET-SOC Code", "Title"]].drop_duplicates("O*NET-SOC Code"),
        left_on="occupation_code", right_on="O*NET-SOC Code", how="left", validate="many_to_one"
    )[["occupation_code", "Title", "similarity"]]

source_code = "15-1252.00"  # Software Developers
for label, matrix in [("RAW COSINE", similarity_df), ("CENTERED COSINE", centered_similarity_df), ("Z-SCORE COSINE", scaled_similarity_df)]:
    print(f"{label} — Top 10")
    display(top_occupations_for_metric(matrix, source_code))

RAW COSINE — Top 10


,occupation_code,Title,similarity
0,15-1243.00,Database Architects,0.994121
1,15-1254.00,Web Developers,0.990058
2,15-1243.01,Data Warehousing Specialists,0.989816
3,15-1242.00,Database Administrators,0.989139
4,15-1211.00,Computer Systems Analysts,0.988962
5,15-1299.07,Blockchain Engineers,0.988664
6,15-1299.02,Geographic Information Systems Technologists a...,0.988194
7,15-1211.01,Health Informatics Specialists,0.987377
8,15-1299.01,Web Administrators,0.986203
9,15-1253.00,Software Quality Assurance Analysts and Testers,0.986198


CENTERED COSINE — Top 10


,occupation_code,Title,similarity
0,15-1243.00,Database Architects,0.932966
1,15-1254.00,Web Developers,0.879307
2,15-1242.00,Database Administrators,0.877220
3,15-1211.00,Computer Systems Analysts,0.868432
4,15-1299.07,Blockchain Engineers,0.863234
5,15-1221.00,Computer and Information Research Scientists,0.855128
6,15-1243.01,Data Warehousing Specialists,0.854422
7,15-1253.00,Software Quality Assurance Analysts and Testers,0.845621
8,15-1299.08,Computer Systems Engineers/Architects,0.845281
9,15-1299.02,Geographic Information Systems Technologists a...,0.843433


Z-SCORE COSINE — Top 10


,occupation_code,Title,similarity
0,15-1243.00,Database Architects,0.936816
1,15-1299.07,Blockchain Engineers,0.893643
2,15-1211.00,Computer Systems Analysts,0.887494
3,15-1254.00,Web Developers,0.875155
4,15-1299.08,Computer Systems Engineers/Architects,0.862635
5,15-1242.00,Database Administrators,0.858294
6,15-1243.01,Data Warehousing Specialists,0.855322
7,15-1221.00,Computer and Information Research Scientists,0.848750
8,15-1253.00,Software Quality Assurance Analysts and Testers,0.841132
9,15-1299.01,Web Administrators,0.837357


In [75]:
# ĐÁNH GIÁ: so sánh thứ hạng raw cosine và z-score cosine trên cùng nghề nguồn.
test_codes = [
    "11-1011.00",  # Chief Executives
    "15-1252.00",  # Software Developers
    "13-2011.00"   # Accountants and Auditors
]

for source_code in test_codes:
    raw_top = similarity_df[source_code].drop(index=source_code).nlargest(10)
    z_top = scaled_similarity_df[source_code].drop(index=source_code).nlargest(10)
    overlap = len(set(raw_top.index) & set(z_top.index))
    final_top, _ = career_transition(source_code, top_n=10)

    print(f"\nNghề nguồn: {source_code}")
    print(f"Số nghề trùng giữa raw và z-score Top 10: {overlap}/10")
    print("Raw cosine Top 10:")
    display(top_occupations_for_metric(similarity_df, source_code, top_n=10))
    print("Z-score cosine Top 10:")
    display(top_occupations_for_metric(scaled_similarity_df, source_code, top_n=10))
    print("Top 10 theo career_transition() chính thức:")
    display(final_top[[
        "occupation_code", "Title", "raw_cosine", "zscore_cosine",
        "mean_absolute_skill_gap", "transition_score"
    ]])

print("\nLưu ý: overlap đo mức đồng thuận thứ hạng, không chứng minh độ chính xác nghề nghiệp.")



Nghề nguồn: 11-1011.00
Số nghề trùng giữa raw và z-score Top 10: 8/10
Raw cosine Top 10:


,occupation_code,Title,similarity
0,11-2022.00,Sales Managers,0.992996
1,11-2011.00,Advertising and Promotions Managers,0.992105
2,11-2021.00,Marketing Managers,0.991565
3,11-3071.04,Supply Chain Managers,0.990553
4,15-1299.09,Information Technology Project Managers,0.988883
5,27-2012.03,Media Programming Directors,0.988729
6,11-1021.00,General and Operations Managers,0.988706
7,11-1011.03,Chief Sustainability Officers,0.988080
8,11-3061.00,Purchasing Managers,0.987671
9,13-1082.00,Project Management Specialists,0.987428


Z-score cosine Top 10:


,occupation_code,Title,similarity
0,11-1021.00,General and Operations Managers,0.949252
1,11-1011.03,Chief Sustainability Officers,0.949052
2,11-2022.00,Sales Managers,0.948935
3,11-3071.04,Supply Chain Managers,0.948037
4,11-2021.00,Marketing Managers,0.939395
5,27-2012.03,Media Programming Directors,0.938831
6,15-1299.09,Information Technology Project Managers,0.935908
7,11-9021.00,Construction Managers,0.929698
8,11-2011.00,Advertising and Promotions Managers,0.929015
9,11-9032.00,"Education Administrators, Kindergarten through...",0.927818


Top 10 theo career_transition() chính thức:


...

Z-score cosine Top 10:


,occupation_code,Title,similarity
0,41-3031.00,"Securities, Commodities, and Financial Service...",0.866427
1,13-1041.08,Customs Brokers,0.848122
2,11-2033.00,Fundraising Managers,0.832433
3,13-1051.00,Cost Estimators,0.827525
4,13-2052.00,Personal Financial Advisors,0.813618
5,13-2031.00,Budget Analysts,0.805476
6,13-1199.06,Online Merchants,0.793709
7,11-3031.03,Investment Fund Managers,0.793589
8,13-1131.00,Fundraisers,0.776957
9,11-2032.00,Public Relations Managers,0.771678


Top 10 theo career_transition() chính thức:


,occupation_code,Title,raw_cosine,zscore_cosine,mean_absolute_skill_gap,transition_score
1,41-3031.00,"Securities, Commodities, and Financial Service...",0.993297,0.866427,0.242000,0.999450
2,13-1041.08,Customs Brokers,0.991213,0.848122,0.275143,0.998350
3,11-2033.00,Fundraising Managers,0.991229,0.832433,0.318571,0.995050
4,13-1199.06,Online Merchants,0.990382,0.793709,0.303714,0.994499
5,13-2031.00,Budget Analysts,0.985763,0.805476,0.314286,0.993949
6,13-2052.00,Personal Financial Advisors,0.988487,0.813618,0.320857,0.993399
7,29-2099.08,Patient Representatives,0.985879,0.752022,0.300857,0.992299
8,17-3011.00,Architectural and Civil Drafters,0.985038,0.754742,0.307143,0.991749
9,13-1051.00,Cost Estimators,0.987482,0.827525,0.337429,0.990649
10,13-1131.00,Fundraisers,0.990242,0.776957,0.338857,0.987349



Lưu ý: overlap đo mức đồng thuận thứ hạng, không chứng minh độ chính xác nghề nghiệp.


## Chạy thử và kiểm tra cuối cùng

Các test bên dưới kiểm tra xếp hạng, phần giải thích và xử lý mã nghề không hợp lệ. Kết quả là kiểm thử kỹ thuật trên các trường hợp mẫu, không thay thế việc đánh giá chất lượng đề xuất bằng dữ liệu thực tế.


In [76]:
import numpy as np
import pandas as pd

# Test 1: Tìm và xếp hạng nghề đích
try:
    source_code = "15-1252.00"  # Software Developers

    top_10, skill_gaps = career_transition(
        source_code=source_code,
        top_n=10
    )

    required_cols = {
        "occupation_code",
        "Title",
        "raw_cosine",
        "zscore_cosine",
        "mean_absolute_skill_gap",
        "transition_score"
    }

    assert isinstance(top_10, pd.DataFrame)
    assert not top_10.empty
    assert len(top_10) <= 10
    assert required_cols.issubset(top_10.columns)
    assert top_10["occupation_code"].is_unique
    assert top_10["occupation_code"].ne(source_code).all()
    assert np.isfinite(
        top_10[
            [
                "raw_cosine",
                "zscore_cosine",
                "mean_absolute_skill_gap",
                "transition_score"
            ]
        ].to_numpy(dtype=float)
    ).all()
    assert top_10["mean_absolute_skill_gap"].ge(0).all()
    assert top_10["transition_score"].between(0, 1).all()
    assert top_10["transition_score"].is_monotonic_decreasing

    print("\n[PASS] Test 1: Career ranking")

    display(
        top_10[
            [
                "occupation_code",
                "Title",
                "raw_cosine",
                "zscore_cosine",
                "mean_absolute_skill_gap",
                "transition_score"
            ]
        ]
    )

except Exception as e:
    print(f"\n[FAIL] Test 1: Career ranking — {type(e).__name__}: {e}")
    raise

# Test 2: Người dùng chọn nghề đích và xem giải thích
try:
    target_code = top_10.iloc[0]["occupation_code"]

    explanation = explain_transition(
        source_code=source_code,
        target_code=target_code,
        top_n=5
    )

    assert isinstance(explanation, dict)
    assert "similarity" in explanation
    assert "mean_absolute_skill_gap" in explanation
    assert "skills_to_focus" in explanation
    assert "transferable_skills" in explanation

    focus = explanation["skills_to_focus"]
    transferable = explanation["transferable_skills"]

    assert isinstance(focus, pd.DataFrame)
    assert isinstance(transferable, pd.DataFrame)
    assert not focus.empty
    assert not transferable.empty
    assert focus["level_gap"].gt(0).all()
    # Transferable skills là các kỹ năng có gap nhỏ nhất; không bắt buộc mọi gap phải bằng 0.
    assert transferable["level_gap"].abs().is_monotonic_increasing
    assert "raw_cosine" in explanation
    assert "zscore_cosine" in explanation

    print("\n[PASS] Test 2: Transition explanation")
    print(f"Source: {source_code}")
    print(f"Target: {target_code}")
    print(f"Similarity: {explanation['similarity']:.4f}")
    print(
        "Mean absolute skill gap: "
        f"{explanation['mean_absolute_skill_gap']:.4f}"
    )

    print("\nSkills to focus on:")
    display(focus)

    print("\nTransferable skills:")
    display(transferable)

except Exception as e:
    print(
        f"\n[FAIL] Test 2: Transition explanation — "
        f"{type(e).__name__}: {e}"
    )
    raise

# Test 3: Mã nghề không hợp lệ phải bị từ chối
try:
    try:
        career_transition(
            source_code="INVALID_CODE",
            top_n=10
        )
    except ValueError as e:
        assert "Occupation code not found" in str(e)
    else:
        raise AssertionError(
            "Hàm không báo ValueError với mã nghề không hợp lệ."
        )

    print("\n[PASS] Test 3: Invalid occupation code")

except Exception as e:
    print(
        f"\n[FAIL] Test 3: Invalid occupation code — "
        f"{type(e).__name__}: {e}"
    )
    raise



[PASS] Test 1: Career ranking


,occupation_code,Title,raw_cosine,zscore_cosine,mean_absolute_skill_gap,transition_score
1,15-1243.00,Database Architects,0.994121,0.936816,0.248000,0.999450
2,15-1299.07,Blockchain Engineers,0.988664,0.893643,0.310286,0.998350
3,15-1254.00,Web Developers,0.990058,0.875155,0.338000,0.996700
4,15-1242.00,Database Administrators,0.989139,0.858294,0.346571,0.995050
5,15-1243.01,Data Warehousing Specialists,0.989816,0.855322,0.372571,0.993949
6,15-1211.00,Computer Systems Analysts,0.988962,0.887494,0.418286,0.992299
7,15-1299.08,Computer Systems Engineers/Architects,0.985526,0.862635,0.417429,0.991749
8,15-1299.01,Web Administrators,0.986203,0.837357,0.386000,0.991749
9,15-1299.05,Information Security Engineers,0.985580,0.834934,0.388571,0.990649
10,15-1253.00,Software Quality Assurance Analysts and Testers,0.986198,0.841132,0.435429,0.988449



[PASS] Test 2: Transition explanation
Source: 15-1252.00
Target: 15-1243.00
Similarity: 0.9941
Mean absolute skill gap: 0.2480

Skills to focus on:


,skill_name,source_level,target_level,level_gap,gap_type
0,Complex Problem Solving,3.50,4.00,0.50,Target requires higher level
1,Instructing,3.12,3.62,0.50,Target requires higher level
2,Writing,3.62,3.88,0.26,Target requires higher level
3,Speaking,3.62,3.88,0.26,Target requires higher level
4,Operations Analysis,3.12,3.38,0.26,Target requires higher level



Transferable skills:


,skill_name,source_level,target_level,level_gap,gap_type
13,Active Learning,3.62,3.62,0.0,Same level
10,Coordination,3.12,3.12,0.0,Same level
11,Negotiation,2.88,2.88,0.0,Same level
14,Programming,4.12,4.12,0.0,Same level
12,Service Orientation,3.00,3.00,0.0,Same level



[PASS] Test 3: Invalid occupation code


## 12.2. Đánh giá Career Recommendation bằng O*NET Related Occupations

Mục tiêu là so sánh danh sách nghề do mô hình skill-based similarity đề xuất với danh sách **O*NET Related Occupations** làm baseline tham chiếu. O*NET Related Occupations được xây dựng dựa trên nhiều yếu tố như công việc thực hiện, kiến thức và tên nghề; vì vậy đây là nguồn tham chiếu độc lập tương đối, không phải ground truth tuyệt đối.

**Câu hỏi đánh giá:** kết quả skill-based similarity có mức độ trùng lặp nào với danh sách nghề liên quan của O*NET, và những trường hợp khác biệt đáng chú ý là gì?

Các chỉ số:
- **Top-K overlap:** số nghề xuất hiện trong cả Top-K của mô hình và Top-K của O*NET.
- **Overlap rate / Recall@K:** số nghề chung chia cho K nghề tham chiếu.
- **Jaccard@K:** số nghề chung chia cho số nghề thuộc hợp của hai danh sách.
- **Similarity distribution:** thống kê raw cosine, z-score cosine và transition score của các nghề được đề xuất; đồng thời so sánh nghề trùng và nghề không trùng với baseline.
- **Recommendation differences:** liệt kê nghề chỉ xuất hiện trong mô hình hoặc chỉ xuất hiện trong baseline để xem xét thủ công.

Không dùng các chỉ số này để khẳng định mô hình “đúng tuyệt đối”. Chúng đo mức đồng thuận với một nguồn tham chiếu có cách xây dựng khác với mô hình.


### Chuẩn bị dữ liệu baseline

Tải tệp **Related Occupations** từ O*NET Database và lưu tại:

`../data/raw/onet/related_occupations.csv`

Nguồn tài liệu chính thức: [O*NET Related Occupations — Data Dictionary](https://www.onetcenter.org/dictionary/current/excel/related_occupations.html).

Tệp có thể có các cột như `O*NET-SOC Code`, `Related O*NET-SOC Code`, `Relatedness Tier`, `Index` (một số bản có thêm cột Title). Nếu tên cột khác, cell kế tiếp sẽ báo các cột đang có để dễ điều chỉnh.

Để so sánh công bằng:
- Top 5 baseline lấy 5 nghề `Primary-Short` đầu tiên.
- Top 10 baseline lấy 10 nghề thuộc nhóm `Primary-Short` và `Primary-Long`, sắp theo `Index`.
- Supplemental occupations (thứ hạng 11–20) không được dùng trong baseline Top 10 chính, nhưng vẫn có thể tham khảo riêng.


In [77]:
# LOAD AND VALIDATE O*NET RELATED OCCUPATIONS
from pathlib import Path

related_path = Path("../data/raw/onet/related_occupations.csv")

if not related_path.exists():
    related_path = Path("../data/raw/onet/Related Occupations.csv")

if not related_path.exists():
    print("CHƯA CHẠY ĐÁNH GIÁ: chưa tìm thấy tệp O*NET Related Occupations.")
    print("Hãy tải tệp từ O*NET và lưu thành ../data/raw/onet/related_occupations.csv")
    related_occupations = None
else:
    try:
        related_occupations = pd.read_csv(related_path)
        if len(related_occupations.columns) == 1:
            related_occupations = pd.read_csv(related_path, sep="\\t")
    except Exception:
        related_occupations = pd.read_csv(related_path, sep="\\t")

    # Chuẩn hóa tên cột để hỗ trợ tên cột từ nhiều định dạng O*NET
    related_occupations.columns = [
        str(c).strip().replace("\\ufeff", "") for c in related_occupations.columns
    ]
    lower_to_original = {c.lower(): c for c in related_occupations.columns}

    def find_column(candidates):
        for candidate in candidates:
            if candidate.lower() in lower_to_original:
                return lower_to_original[candidate.lower()]
        return None

    source_col = find_column(["O*NET-SOC Code", "onetsoc_code"])
    related_col = find_column(["Related O*NET-SOC Code", "related_onetsoc_code"])
    tier_col = find_column(["Relatedness Tier", "relatedness_tier"])
    index_col = find_column(["Index", "related_index"])

    required = {
        "source occupation code": source_col,
        "related occupation code": related_col,
        "relatedness tier": tier_col,
        "relatedness index": index_col,
    }
    missing = [name for name, col in required.items() if col is None]

    if missing:
        raise ValueError(
            "Thiếu cột bắt buộc trong O*NET Related Occupations: "
            + ", ".join(missing)
            + f". Các cột hiện có: {related_occupations.columns.tolist()}"
        )

    related_occupations = related_occupations.rename(columns={
        source_col: "source_code",
        related_col: "related_code",
        tier_col: "relatedness_tier",
        index_col: "related_index",
    })

    related_occupations["source_code"] = related_occupations["source_code"].astype(str).str.strip()
    related_occupations["related_code"] = related_occupations["related_code"].astype(str).str.strip()
    related_occupations["relatedness_tier"] = related_occupations["relatedness_tier"].astype(str).str.strip()
    related_occupations["related_index"] = pd.to_numeric(
        related_occupations["related_index"], errors="coerce"
    )

    related_occupations = related_occupations.dropna(
        subset=["source_code", "related_code", "related_index"]
    ).drop_duplicates(subset=["source_code", "related_code"])

    # Chỉ so sánh các nghề có trong cả ma trận kỹ năng và dữ liệu baseline.
    related_occupations = related_occupations[
        related_occupations["source_code"].isin(skill_matrix.index)
        & related_occupations["related_code"].isin(skill_matrix.index)
        & (related_occupations["source_code"] != related_occupations["related_code"])
    ].copy()

    print("O*NET Related Occupations: LOAD PASS")
    print("Số dòng sau làm sạch và khớp mã nghề:", len(related_occupations))
    print("Số nghề nguồn có baseline:", related_occupations["source_code"].nunique())
    display(related_occupations.head())


O*NET Related Occupations: LOAD PASS
Số dòng sau làm sạch và khớp mã nghề: 17776
Số nghề nguồn có baseline: 910


,source_code,Title,related_code,Related Title,relatedness_tier,related_index
0,11-1011.00,Chief Executives,11-1021.00,General and Operations Managers,Primary-Short,1
1,11-1011.00,Chief Executives,13-1111.00,Management Analysts,Primary-Short,2
2,11-1011.00,Chief Executives,11-3031.01,Treasurers and Controllers,Primary-Short,3
3,11-1011.00,Chief Executives,11-9151.00,Social and Community Service Managers,Primary-Short,4
4,11-1011.00,Chief Executives,11-2032.00,Public Relations Managers,Primary-Short,5


In [78]:
# CAREER RECOMMENDATION EVALUATION: O*NET BASELINE VS SKILL SIMILARITY
# Chạy cell này sau khi related_occupations đã được nạp thành công ở cell trên.

if related_occupations is None:
    print("Bỏ qua đánh giá: hãy thêm tệp O*NET Related Occupations rồi chạy lại cell load.")
else:
    evaluation_sources = [
        "15-1252.00",  # Software Developers
        "15-1243.00",  # Database Architects
        "11-1011.00",  # Chief Executives
        "13-2011.00",  # Accountants and Auditors
    ]
    evaluation_sources = [
        code for code in evaluation_sources
        if code in skill_matrix.index
        and code in set(related_occupations["source_code"])
    ]

    summary_rows = []
    detail_rows = []
    distribution_rows = []

    for source_code in evaluation_sources:
        baseline = (
            related_occupations[related_occupations["source_code"] == source_code]
            .sort_values("related_index")
        )

        # Baseline top-5 = Primary-Short (first five); top-10 = first ten primary.
        primary = baseline[
            baseline["relatedness_tier"].str.lower().isin(["primary-short", "primary-long"])
        ].sort_values("related_index")
        baseline_top5 = primary.head(5)["related_code"].tolist()
        baseline_top10 = primary.head(10)["related_code"].tolist()

        model_ranked, _ = career_transition(source_code=source_code, top_n=20)
        model_top5 = model_ranked.head(5)["occupation_code"].tolist()
        model_top10 = model_ranked.head(10)["occupation_code"].tolist()

        for k, model_list, baseline_list in [
            (5, model_top5, baseline_top5),
            (10, model_top10, baseline_top10),
        ]:
            model_set, baseline_set = set(model_list), set(baseline_list)
            common = sorted(model_set & baseline_set)
            union = model_set | baseline_set

            summary_rows.append({
                "source_code": source_code,
                "source_title": (
                    occupation_data.loc[
                        occupation_data["O*NET-SOC Code"] == source_code, "Title"
                    ].iloc[0]
                    if (occupation_data["O*NET-SOC Code"] == source_code).any()
                    else source_code
                ),
                "K": k,
                "model_count": len(model_set),
                "onet_count": len(baseline_set),
                "overlap_count": len(common),
                "overlap_rate_recall_at_k": len(common) / len(baseline_set) if baseline_set else np.nan,
                "jaccard_at_k": len(common) / len(union) if union else np.nan,
            })

            # Candidate differences: which occupations appear only in one list?
            for code in sorted(model_set - baseline_set):
                row = model_ranked[model_ranked["occupation_code"] == code].iloc[0]
                detail_rows.append({
                    "source_code": source_code, "K": k,
                    "difference_type": "Model only",
                    "occupation_code": code, "Title": row.get("Title"),
                    "model_rank": int(model_ranked.index[model_ranked["occupation_code"] == code][0]) + 1,
                    "relatedness_tier": None,
                    "related_index": np.nan,
                    "zscore_cosine": row.get("zscore_cosine"),
                    "transition_score": row.get("transition_score"),
                })

            for code in sorted(baseline_set - model_set):
                b_row = baseline[baseline["related_code"] == code].sort_values("related_index").iloc[0]
                title_col = "Related Title" if "Related Title" in baseline.columns else (
                    "related_title" if "related_title" in baseline.columns else None
                )
                title = b_row[title_col] if title_col else (
                    occupation_data.loc[
                        occupation_data["O*NET-SOC Code"] == code, "Title"
                    ].iloc[0]
                    if (occupation_data["O*NET-SOC Code"] == code).any()
                    else None
                )
                detail_rows.append({
                    "source_code": source_code, "K": k,
                    "difference_type": "O*NET only",
                    "occupation_code": code, "Title": title,
                    "model_rank": np.nan,
                    "relatedness_tier": b_row["relatedness_tier"],
                    "related_index": b_row["related_index"],
                    "zscore_cosine": (
                        float(scaled_similarity_df.loc[source_code, code])
                        if code in scaled_similarity_df.columns else np.nan
                    ),
                    "transition_score": np.nan,
                })

        # Distribution: compare model top-10 candidates that overlap / do not overlap with O*NET primary top-10.
        baseline_set10 = set(baseline_top10)
        for _, row in model_ranked.head(10).iterrows():
            distribution_rows.append({
                "source_code": source_code,
                "occupation_code": row["occupation_code"],
                "Title": row.get("Title"),
                "in_onet_primary_top10": row["occupation_code"] in baseline_set10,
                "raw_cosine": row.get("raw_cosine"),
                "zscore_cosine": row.get("zscore_cosine"),
                "mean_absolute_skill_gap": row.get("mean_absolute_skill_gap"),
                "transition_score": row.get("transition_score"),
            })

    evaluation_summary = pd.DataFrame(summary_rows)
    evaluation_differences = pd.DataFrame(detail_rows)
    evaluation_distribution = pd.DataFrame(distribution_rows)

    print("=== TOP-K OVERLAP SUMMARY ===")
    display(evaluation_summary.round(3))

    print("\\n=== SIMILARITY DISTRIBUTION: MODEL TOP-10 ===")
    if not evaluation_distribution.empty:
        display(
            evaluation_distribution.groupby("in_onet_primary_top10")[
                ["raw_cosine", "zscore_cosine", "mean_absolute_skill_gap", "transition_score"]
            ].agg(["count", "mean", "median", "min", "max"]).round(4)
        )

    print("\\n=== RECOMMENDATION DIFFERENCES ===")
    if evaluation_differences.empty:
        print("Không có khác biệt trong các Top-K được so sánh.")
    else:
        display(evaluation_differences.sort_values(
            ["source_code", "K", "difference_type", "model_rank"],
            na_position="last"
        ).reset_index(drop=True))

    # Overall summary across evaluated source occupations
    if not evaluation_summary.empty:
        print("\\n=== OVERALL MEAN ===")
        display(
            evaluation_summary.groupby("K")[
                ["overlap_count", "overlap_rate_recall_at_k", "jaccard_at_k"]
            ].mean().round(3)
        )


=== TOP-K OVERLAP SUMMARY ===


,source_code,source_title,K,model_count,onet_count,overlap_count,overlap_rate_recall_at_k,jaccard_at_k
0,15-1252.00,Software Developers,5,5,5,1,0.200,0.111
1,15-1252.00,Software Developers,10,10,10,7,0.700,0.538
2,15-1243.00,Database Architects,5,5,5,3,0.600,0.429
3,15-1243.00,Database Architects,10,10,10,6,0.600,0.429
4,11-1011.00,Chief Executives,5,5,5,1,0.200,0.111
5,11-1011.00,Chief Executives,10,10,9,2,0.222,0.118
6,13-2011.00,Accountants and Auditors,5,5,5,0,0.000,0.000
7,13-2011.00,Accountants and Auditors,10,10,9,2,0.222,0.118


\n=== SIMILARITY DISTRIBUTION: MODEL TOP-10 ===


raw_cosine                                  \
                           count    mean  median     min     max   
in_onet_primary_top10                                              
False                         23  0.9885  0.9889  0.9837  0.9933   
True                          17  0.9887  0.9885  0.9834  0.9946   

                      zscore_cosine                                  \
                              count    mean  median     min     max   
in_onet_primary_top10                                                 
False                            23  0.8667  0.8664  0.7520  0.9489   
True                             17  0.8813  0.8752  0.8055  0.9491   

                      mean_absolute_skill_gap                                  \
                                        count    mean  median     min     max   
in_onet_primary_top10                                                           
False                                      23  0.4441  0.3540  0.2420  0.7400   
True                                       17  0.3796  0.3466  0.2363  0.6726   

                      transition_score                                  
                                 count    mean  median     min     max  
in_onet_primary_top10                                                   
False                               23  0.9928  0.9923  0.9873  0.9994  
True                                17  0.9940  0.9939  0.9873  0.9994

\n=== RECOMMENDATION DIFFERENCES ===


,source_code,K,difference_type,occupation_code,Title,model_rank,relatedness_tier,related_index,zscore_cosine,transition_score
0,11-1011.00,5,Model only,11-2022.00,Sales Managers,2.0,None,NaN,0.948935,0.997800
1,11-1011.00,5,Model only,11-1011.03,Chief Sustainability Officers,3.0,None,NaN,0.949052,0.995050
2,11-1011.00,5,Model only,11-2021.00,Marketing Managers,4.0,None,NaN,0.939395,0.995050
3,11-1011.00,5,Model only,11-9032.00,"Education Administrators, Kindergarten through...",5.0,None,NaN,0.927818,0.994499
4,11-1011.00,5,O*NET only,11-1021.00,General and Operations Managers,NaN,Primary-Short,1.0,0.949252,NaN
...,...,...,...,...,...,...,...,...,...,...
69,15-1252.00,10,Model only,15-1243.01,Data Warehousing Specialists,6.0,None,NaN,0.855322,0.993949
70,15-1252.00,10,Model only,15-1299.01,Web Administrators,9.0,None,NaN,0.837357,0.991749
71,15-1252.00,10,O*NET only,15-1241.00,Computer Network Architects,NaN,Primary-Long,8.0,0.822316,NaN
72,15-1252.00,10,O*NET only,15-1251.00,Computer Programmers,NaN,Primary-Short,4.0,0.800024,NaN


\n=== OVERALL MEAN ===


,overlap_count,overlap_rate_recall_at_k,jaccard_at_k
K,,,
5,1.25,0.250,0.163
10,4.25,0.436,0.301


### Cách diễn giải kết quả

- **Overlap cao** cho thấy hai phương pháp thường đề xuất một số nghề giống nhau; không tự động chứng minh đề xuất đúng.
- **Overlap thấp** không nhất thiết là lỗi. O*NET Related Occupations dựa trên nhiều yếu tố (công việc, kiến thức và tên nghề), trong khi mô hình này chỉ dùng vector kỹ năng LV.
- Xem bảng `RECOMMENDATION DIFFERENCES` để chọn một số trường hợp khác biệt và kiểm tra thủ công xem mô hình đang tìm ra hướng chuyển đổi nào mà baseline không liệt kê, hoặc bỏ qua nghề nào trong baseline.
- Xem phân phối similarity của các nghề trùng và không trùng với baseline. Không nên kết luận chỉ từ mean/median; hãy đọc cả từng trường hợp và đặc điểm nghề.
- Ghi rõ số nghề nguồn thực sự được đánh giá. Mã nghề không có trong cả hai nguồn sẽ bị loại khỏi đánh giá.

**Lưu ý về tính độc lập:** O*NET Related Occupations là baseline tham chiếu, không phải nhãn ground-truth hoàn hảo. Báo cáo nên mô tả đây là phép so sánh mức độ đồng thuận, không phải đo accuracy tuyệt đối.
